# Fingerprint Blood Group Detection Demo

This notebook demonstrates how to use the trained model to predict blood groups from fingerprint images.

In [ ]:
import torch
from models.hybrid_model import HybridMultiModalNet
from features.handcrafted import extract_all
from data.transforms import val_test_transforms
from PIL import Image
import matplotlib.pyplot as plt
from config import ABO_CLASSES, RH_CLASSES

# Reverse mappings
ABO_LABELS = {v: k for k, v in ABO_CLASSES.items()}
RH_LABELS = {v: k for k, v in RH_CLASSES.items()}

In [ ]:
# Load the trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HybridMultiModalNet()
# Load checkpoint (replace with actual path)
# checkpoint = torch.load('outputs/checkpoints/model_epoch_X_loss_X.XXXX.pth', map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()
print("Model loaded successfully")

In [ ]:
# Function to predict blood group
def predict_blood_group(image_path):
    # Load and transform image
    image = Image.open(image_path).convert('RGB')
    plt.imshow(image)
    plt.title('Fingerprint Image')
    plt.show()
    
    image_tensor = val_test_transforms(image).unsqueeze(0).to(device)
    
    # Extract handcrafted features
    handcrafted = extract_all(image_path)
    handcrafted_tensor = torch.tensor(handcrafted, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        abo_logits, rh_logits = model(image_tensor, handcrafted_tensor)
        
        abo_probs = torch.softmax(abo_logits, dim=1)
        rh_probs = torch.softmax(rh_logits, dim=1)
        
        abo_pred_idx = torch.argmax(abo_probs, dim=1).item()
        rh_pred_idx = torch.argmax(rh_probs, dim=1).item()
        
        abo_pred = ABO_LABELS[abo_pred_idx]
        rh_pred = RH_LABELS[rh_pred_idx]
        abo_conf = abo_probs[0, abo_pred_idx].item()
        rh_conf = rh_probs[0, rh_pred_idx].item()
    
    print(f"Predicted ABO Group: {abo_pred} (confidence: {abo_conf:.4f})")
    print(f"Predicted Rh Factor: {rh_pred} (confidence: {rh_conf:.4f})")
    print(f"Full Blood Group: {abo_pred}{rh_pred}")
    
    return abo_pred, rh_pred

In [ ]:
# Example usage (replace with actual image path)
# predict_blood_group('dataset/dataset_blood_group/A+/sample.jpg')